In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.metrics         import (accuracy_score, f1_score, precision_score,
                                     recall_score, roc_auc_score,
                                     average_precision_score, confusion_matrix,
                                     precision_recall_curve, roc_curve)
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)
print("All libraries loaded!")

In [ ]:
data = pd.read_csv("smart_grid_stability_augmented.csv")

data = data.rename(columns={
    "tau1" : "reaction_producer",
    "tau2" : "reaction_consumer1",
    "tau3" : "reaction_consumer2",
    "tau4" : "reaction_consumer3",
    "p1"   : "power_producer",
    "p2"   : "power_consumer1",
    "p3"   : "power_consumer2",
    "p4"   : "power_consumer3",
    "g1"   : "price_sensitivity_producer",
    "g2"   : "price_sensitivity_consumer1",
    "g3"   : "price_sensitivity_consumer2",
    "g4"   : "price_sensitivity_consumer3",
    "stab" : "stability_score",
    "stabf": "grid_status",
})

print(f"Rows    : {len(data):,}")
print(f"Columns : {list(data.columns)}")
data.head()

In [ ]:
print("Missing values  :", data.isnull().sum().sum(),  "← should be 0")
print("Duplicate rows  :", data.duplicated().sum(),    "← should be 0")
print()
print("Label counts:")
print(data["grid_status"].value_counts())

In [ ]:
plt.figure(figsize=(16, 10))
plt.suptitle("EDA — Understanding the Grid Stability Data", fontsize=14)

# ── Chart 1: Stable vs Unstable count ───────────────────
plt.subplot(2, 3, 1)
counts = data["grid_status"].value_counts()
plt.bar(counts.index, counts.values, color=["steelblue", "tomato"])
plt.title("Stable vs Unstable Count")
plt.ylabel("Number of rows")
for i, val in enumerate(counts.values):
    plt.text(i, val + 200, f"{val:,}", ha="center", fontweight="bold")

In [ ]:
# ── Chart 2: Stability score by class ───────────────────
# Negative score = unstable, positive = stable
plt.subplot(2, 3, 2)
sns.histplot(data[data["grid_status"]=="stable"]["stability_score"],
             bins=50, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["stability_score"],
             bins=50, color="tomato", alpha=0.6, label="Unstable")
plt.axvline(0, color="black", linestyle="--", label="Boundary = 0")
plt.title("Stability Score by Class")
plt.xlabel("stability_score")
plt.legend()

# ── Chart 3: Reaction producer by class ─────────────────
plt.subplot(2, 3, 3)
sns.histplot(data[data["grid_status"]=="stable"]["reaction_producer"],
             bins=40, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["reaction_producer"],
             bins=40, color="tomato", alpha=0.6, label="Unstable")
plt.title("Reaction Producer by Class")
plt.xlabel("reaction_producer")
plt.legend()


In [ ]:
# ── Chart 4: Power producer by class ────────────────────
plt.subplot(2, 3, 4)
sns.histplot(data[data["grid_status"]=="stable"]["power_producer"],
             bins=40, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["power_producer"],
             bins=40, color="tomato", alpha=0.6, label="Unstable")
plt.title("Power Producer by Class")
plt.xlabel("power_producer")
plt.legend()

# ── Chart 5: Reaction consumer1 by class ────────────────
plt.subplot(2, 3, 5)
sns.histplot(data[data["grid_status"]=="stable"]["reaction_consumer1"],
             bins=40, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["reaction_consumer1"],
             bins=40, color="tomato", alpha=0.6, label="Unstable")
plt.title("Reaction Consumer1 by Class")
plt.xlabel("reaction_consumer1")
plt.legend()

In [ ]:
# ── Chart 6: Correlation heatmap ─────────────────────────
plt.subplot(2, 3, 6)
cols_for_corr = ["reaction_producer", "power_producer",
                 "price_sensitivity_producer", "stability_score"]
corr = data[cols_for_corr].corr().round(2)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5)
plt.title("Correlation Heatmap")

plt.tight_layout()
plt.savefig("outputs/01_eda.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/01_eda.png")

In [ ]:
input_features = [
    "reaction_producer",  "reaction_consumer1", "reaction_consumer2", "reaction_consumer3",
    "power_producer",     "power_consumer1",    "power_consumer2",    "power_consumer3",
    "price_sensitivity_producer",  "price_sensitivity_consumer1",
    "price_sensitivity_consumer2", "price_sensitivity_consumer3",
]

X = data[input_features]
y = (data["grid_status"] == "unstable").astype(int)   # 1 = unstable, 0 = stable

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training rows : {len(X_train):,}")
print(f"Testing rows  : {len(X_test):,}")

In [ ]:
# Create the 5 models
log_reg  = LogisticRegression(max_iter=1000, random_state=42)
dec_tree = DecisionTreeClassifier(max_depth=8, random_state=42)
ran_for  = RandomForestClassifier(n_estimators=100, random_state=42)
grad_bst = GradientBoostingClassifier(n_estimators=100, random_state=42)
knn      = KNeighborsClassifier(n_neighbors=7)

# Train each model
# Logistic Regression and KNN need scaled data
# Tree models work on original data
log_reg.fit(X_train_scaled, y_train)
dec_tree.fit(X_train, y_train)
ran_for.fit(X_train, y_train)
grad_bst.fit(X_train, y_train)
knn.fit(X_train_scaled, y_train)

print("All 5 models trained!")

In [ ]:
# Get predictions (0 or 1) and probabilities (0.0 to 1.0) for each model
pred_log  = log_reg.predict(X_test_scaled)
pred_dec  = dec_tree.predict(X_test)
pred_ran  = ran_for.predict(X_test)
pred_grad = grad_bst.predict(X_test)
pred_knn  = knn.predict(X_test_scaled)

prob_log  = log_reg.predict_proba(X_test_scaled)[:, 1]
prob_dec  = dec_tree.predict_proba(X_test)[:, 1]
prob_ran  = ran_for.predict_proba(X_test)[:, 1]
prob_grad = grad_bst.predict_proba(X_test)[:, 1]
prob_knn  = knn.predict_proba(X_test_scaled)[:, 1]

# Score each model
model_names = ["Logistic Regression", "Decision Tree",
               "Random Forest", "Gradient Boosting", "K-Nearest Neighbors"]
all_preds   = [pred_log,  pred_dec,  pred_ran,  pred_grad,  pred_knn]
all_probas  = [prob_log,  prob_dec,  prob_ran,  prob_grad,  prob_knn]

acc_scores  = [accuracy_score(y_test, p)          for p in all_preds]
f1_scores   = [f1_score(y_test, p)                for p in all_preds]
prec_scores = [precision_score(y_test, p)         for p in all_preds]
rec_scores  = [recall_score(y_test, p)            for p in all_preds]
roc_scores  = [roc_auc_score(y_test, p)           for p in all_probas]
ap_scores   = [average_precision_score(y_test, p) for p in all_probas]

print(f"{'Model':<25} {'Accuracy':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'ROC-AUC':>9}")
print("-" * 72)
for i, name in enumerate(model_names):
    print(f"{name:<25} {acc_scores[i]:>10.2%} {f1_scores[i]:>8.4f}"
          f" {prec_scores[i]:>10.4f} {rec_scores[i]:>8.4f} {roc_scores[i]:>9.4f}")

best_idx  = f1_scores.index(max(f1_scores))
best_name = model_names[best_idx]
print(f"\nBest model: {best_name}  (F1 = {max(f1_scores):.4f})")

In [ ]:
colors = ["steelblue", "seagreen", "darkorange", "mediumpurple", "tomato"]

plt.figure(figsize=(18, 5))
plt.suptitle("Model Comparison — All 5 Models", fontsize=14)

# Accuracy
plt.subplot(1, 4, 1)
plt.bar(model_names, acc_scores, color=colors, alpha=0.85)
plt.title("Accuracy")
plt.ylim(0, 1.15)
plt.xticks(range(len(model_names)), [n.replace(" ", "\n") for n in model_names], fontsize=8)
for i, val in enumerate(acc_scores):
    plt.text(i, val + 0.01, f"{val:.3f}", ha="center", fontsize=8, fontweight="bold")

# F1 Score
plt.subplot(1, 4, 2)
plt.bar(model_names, f1_scores, color=colors, alpha=0.85)
plt.title("F1 Score")
plt.ylim(0, 1.15)
plt.xticks(range(len(model_names)), [n.replace(" ", "\n") for n in model_names], fontsize=8)
for i, val in enumerate(f1_scores):
    plt.text(i, val + 0.01, f"{val:.3f}", ha="center", fontsize=8, fontweight="bold")

# Precision
plt.subplot(1, 4, 3)
plt.bar(model_names, prec_scores, color=colors, alpha=0.85)
plt.title("Precision")
plt.ylim(0, 1.15)
plt.xticks(range(len(model_names)), [n.replace(" ", "\n") for n in model_names], fontsize=8)
for i, val in enumerate(prec_scores):
    plt.text(i, val + 0.01, f"{val:.3f}", ha="center", fontsize=8, fontweight="bold")

# Recall
plt.subplot(1, 4, 4)
plt.bar(model_names, rec_scores, color=colors, alpha=0.85)
plt.title("Recall")
plt.ylim(0, 1.15)
plt.xticks(range(len(model_names)), [n.replace(" ", "\n") for n in model_names], fontsize=8)
for i, val in enumerate(rec_scores):
    plt.text(i, val + 0.01, f"{val:.3f}", ha="center", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.savefig("outputs/02_model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/02_model_comparison.png")

In [ ]:
plt.figure(figsize=(14, 6))
plt.suptitle("Precision-Recall & ROC Curves", fontsize=14)

# ── Precision-Recall curves ───────────────────────────────
plt.subplot(1, 2, 1)
plt.axhline(y_test.mean(), color="gray", linestyle="--", label=f"Random ({y_test.mean():.2f})")

prec, rec, _ = precision_recall_curve(y_test, prob_log)
plt.plot(rec, prec, color="steelblue",    lw=1.5, label=f"Logistic Reg  (AP={ap_scores[0]:.3f})")

prec, rec, _ = precision_recall_curve(y_test, prob_dec)
plt.plot(rec, prec, color="seagreen",     lw=1.5, label=f"Decision Tree (AP={ap_scores[1]:.3f})")

prec, rec, _ = precision_recall_curve(y_test, prob_ran)
plt.plot(rec, prec, color="darkorange",   lw=2.5, label=f"Random Forest (AP={ap_scores[2]:.3f})")

prec, rec, _ = precision_recall_curve(y_test, prob_grad)
plt.plot(rec, prec, color="mediumpurple", lw=1.5, label=f"Grad Boosting (AP={ap_scores[3]:.3f})")

prec, rec, _ = precision_recall_curve(y_test, prob_knn)
plt.plot(rec, prec, color="tomato",       lw=1.5, label=f"KNN           (AP={ap_scores[4]:.3f})")

plt.title("Precision-Recall Curve\n(top-right corner = best)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(fontsize=8)

# ── ROC curves ────────────────────────────────────────────
plt.subplot(1, 2, 2)
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random")

fpr, tpr, _ = roc_curve(y_test, prob_log)
plt.plot(fpr, tpr, color="steelblue",    lw=1.5, label=f"Logistic Reg  (AUC={roc_scores[0]:.3f})")

fpr, tpr, _ = roc_curve(y_test, prob_dec)
plt.plot(fpr, tpr, color="seagreen",     lw=1.5, label=f"Decision Tree (AUC={roc_scores[1]:.3f})")

fpr, tpr, _ = roc_curve(y_test, prob_ran)
plt.plot(fpr, tpr, color="darkorange",   lw=2.5, label=f"Random Forest (AUC={roc_scores[2]:.3f})")

fpr, tpr, _ = roc_curve(y_test, prob_grad)
plt.plot(fpr, tpr, color="mediumpurple", lw=1.5, label=f"Grad Boosting (AUC={roc_scores[3]:.3f})")

fpr, tpr, _ = roc_curve(y_test, prob_knn)
plt.plot(fpr, tpr, color="tomato",       lw=1.5, label=f"KNN           (AUC={roc_scores[4]:.3f})")

plt.title("ROC Curve\n(top-left corner = best)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(fontsize=8)

plt.tight_layout()
plt.savefig("outputs/03_pr_roc_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/03_pr_roc_curves.png")

In [ ]:
plt.figure(figsize=(14, 6))
plt.suptitle(f"Best Model: {best_name}", fontsize=14)

# ── Confusion Matrix ──────────────────────────────────────
plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, all_preds[best_idx])
sns.heatmap(cm, annot=True, fmt=",", cmap="Blues",
            xticklabels=["Predicted Stable", "Predicted Unstable"],
            yticklabels=["Actual Stable", "Actual Unstable"],
            linewidths=0.5, annot_kws={"size": 14})
plt.title("Confusion Matrix")

# ── Feature Importance ────────────────────────────────────
plt.subplot(1, 2, 2)
best_model_obj = [log_reg, dec_tree, ran_for, grad_bst, knn][best_idx]

if hasattr(best_model_obj, "feature_importances_"):
    importance = best_model_obj.feature_importances_
else:
    importance = abs(best_model_obj.coef_[0])   # for Logistic Regression

# Sort smallest → largest so most important is at the top
sorted_idx   = np.argsort(importance)
sorted_feats = [input_features[i] for i in sorted_idx]
sorted_vals  = importance[sorted_idx]

bar_colors = ["steelblue" if "reaction" in f
              else "tomato" if "power" in f
              else "seagreen" for f in sorted_feats]

plt.barh(sorted_feats, sorted_vals, color=bar_colors, alpha=0.85)
plt.title("Feature Importance")
plt.xlabel("Importance score")
plt.legend(handles=[
    Patch(color="steelblue", label="Reaction times"),
    Patch(color="tomato",    label="Power values"),
    Patch(color="seagreen",  label="Price sensitivity"),
])

plt.tight_layout()
plt.savefig("outputs/04_best_model.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/04_best_model.png")

In [ ]:
lag_cols    = ["reaction_producer", "reaction_consumer1", "power_producer", "power_consumer1"]
seq_lengths = [1, 2, 3, 4, 6, 8, 10, 12]
seq_f1s     = []
seq_rocs    = []

print(f"{'Seq Len':>8}  {'Features':>10}  {'F1 Score':>10}  {'ROC-AUC':>10}")
print("-" * 45)

for seq_len in seq_lengths:

    temp = data[input_features + ["grid_status"]].copy()

    if seq_len > 1:
        for lag in range(1, seq_len):
            for col in lag_cols:
                temp[f"{col}_lag{lag}"] = temp[col].shift(lag)
        temp = temp.dropna()

    feat_cols = [c for c in temp.columns if c != "grid_status"]
    Xs = temp[feat_cols].values
    ys = (temp["grid_status"] == "unstable").astype(int).values

    Xtr, Xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, random_state=42, stratify=ys)
    sc  = StandardScaler()
    Xtr = sc.fit_transform(Xtr)
    Xte = sc.transform(Xte)

    clf = RandomForestClassifier(n_estimators=80, random_state=42)
    clf.fit(Xtr, ytr)

    yp   = clf.predict(Xte)
    yprb = clf.predict_proba(Xte)[:, 1]

    f1  = f1_score(yte, yp)
    roc = roc_auc_score(yte, yprb)
    seq_f1s.append(f1)
    seq_rocs.append(roc)

    print(f"{seq_len:>8}  {Xs.shape[1]:>10}  {f1:>10.4f}  {roc:>10.4f}")

best_seq = seq_lengths[seq_f1s.index(max(seq_f1s))]
print(f"\nBest sequence length: {best_seq}  (F1 = {max(seq_f1s):.4f})") 